# Solar filament segmentation: model averaging + stricter keep rule (Charan)

No training. Loads three trained checkpoints from earlier kernels, predicts with 8-view D4 TTA, averages probability maps, and tunes a keep rule on the 7-day-split validation set:

* candidates = connected components of `p > t_lo` inside the disk, optionally after a morphological closing that bridges short gaps between fragments
* keep a candidate when `peak >= t_hi`, `mean >= min_mean`, `area >= min_area`

Every model has lost 0.03-0.06 PQ from validation to the public board. `fseg_unet/ensemble.py` models this as sparser test labels (rate `r`), fits `r` on two past submissions, and also picks the rule with the best *estimated public* PQ.

| Model | Kernel | Val split | Past public PQ |
|---|---|---|---|
| `unet_week` U-Net r34 | `filament-unet-r34-week` | 7-day (honest) | 0.35 |
| `meshnet_week` MeshNet 2D | `filament-meshnet-week` | 7-day (honest) | 0.33 |
| `unet_r1` U-Net r34 run 1 | `filament-unet-r34` | Sheikh's split (overlaps this val set) | 0.36 |

**Settings:** GPU T4, Internet on, competition data + the three kernels above as inputs.

In [ ]:
# 1. Setup
import os, subprocess, sys, json
from pathlib import Path

BRANCH = "charan/ensemble"
REPO = "https://github.com/hsfcoulibaly/aml_project.git"
SRC = Path("/kaggle/working/aml_project")
if not SRC.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(SRC)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "pycocotools"], check=True)
CODE = SRC / "Charan" / "unet"
os.chdir(CODE); sys.path.insert(0, str(CODE))
os.environ["UNET_WORK_DIR"] = "/kaggle/working/unet"
WORK = Path(os.environ["UNET_WORK_DIR"])
print(subprocess.run(["git", "-C", str(SRC), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Find the checkpoints in the attached kernel outputs
def find_best(slug):
    hits = [p for p in Path("/kaggle/input").rglob("best.pt") if p.parent.name == "unet_results" and slug in p.parts]
    assert len(hits) == 1, (slug, hits)
    return hits[0]

W = {"unet_week": find_best("filament-unet-r34-week"),
     "meshnet_week": find_best("filament-meshnet-week"),
     "unet_r1": find_best("filament-unet-r34")}
for k, v in W.items(): print(k, v)

In [ ]:
# 3. Prepare 1024 px inputs + references on the 7-day block split (~5 min)
!python scripts/prepare.py --size 1024 --workers 4 --split splits/week_split.json

In [ ]:
# 4. Predict, average, sweep keep rules, write submissions
OUT = Path("/kaggle/working/ensemble")
!python scripts/ensemble.py \
    --model unet_week={W['unet_week']} --model meshnet_week={W['meshnet_week']} --model unet_r1={W['unet_r1']} \
    --leaky unet_r1 \
    --calib unet_week:0.7:400:0.35 --calib meshnet_week:0.7:400:0.33 \
    --blend ens2=unet_week:1,meshnet_week:1 --blend ens2w=unet_week:2,meshnet_week:1 \
    --blend ens3=unet_week:1,meshnet_week:1,unet_r1:1 --blend r1=unet_r1:1 \
    --rules-from ens3=ens2 --rules-from r1=unet_week \
    --tta d4 --workers 4 --out-dir {OUT}

In [ ]:
# 5. Summary + clean up the large prepared data
import pandas as pd
s = json.load(open(OUT / "summary.json"))
print("r =", s["r"])
rows = []
for b, d in s["blends"].items():
    for tag in ("pq", "lb"):
        if tag in d:
            rows.append({"blend": b, "pick": tag, **d[tag]["rule"], "val_pq": d[tag]["val"]["pq"], "pq_lb_est": d[tag]["pq_lb_est"],
                         "precision": d[tag]["val"]["precision"], "recall": d[tag]["val"]["recall"],
                         "split": d[tag]["val"]["split_rate"], "merge": d[tag]["val"]["merge_rate"], "test_rows": d.get("test_rows", {}).get(tag)})
        elif "rules_from" in d:
            rows.append({"blend": b, "pick": tag, "rules_from": d["rules_from"], "test_rows": d.get("test_rows", {}).get(tag)})
display(pd.DataFrame(rows))
import shutil
shutil.rmtree(WORK / "images", ignore_errors=True); shutil.rmtree(WORK / "masks", ignore_errors=True)
shutil.rmtree(SRC, ignore_errors=True)
print(sorted(p.name for p in OUT.iterdir()))